## Facebook post generator — creation notebook

Builds the HTML/CSS Facebook-style post via `post_generator_all_visible_emojis.py`
(no overlapping reactions; comment button in place of dislike; nr. of shares/comments shown).

**Where this notebook's output actually goes today:**

| What | Status |
|---|---|
| Main study (remy-ashford / dr-remy-ashford), all conditions | Superseded — use `utils/generate_profile_posts.py --slug <profile>` instead. Verified byte-identical to what this notebook produced. |
| Simple-plot pilot (baseline / realistic / likes_only_noise) | **Still the only source.** No `.py` script covers the simplified chart pool. `likes_only` dropped (2026-09-18) -- not part of the thesis, matches the bigfont pilot which never had it. |
| Phase 1 profile-variant benchmarking (news outlets) | Already fully generated and tracked (12/12 files) — kept below as a historical record, not something to re-run. |

The simple-plot pilot's stimuli (3,600 HTML + 3,600 PNG) are already generated and tracked, so
none of the cells below need to run again for a normal clone — they document how the pilot data
was built, and are the reference if it is ever extended or regenerated.

In [ ]:
BASE_DIR = Path().resolve().parent

## Main study — superseded

Use `python3 utils/generate_profile_posts.py` (default `--slug remy-ashford`) instead. See that script's own docstring for the authority-condition (`dr-remy-ashford`) invocation, `--check`, and `--recover-charts`.

## Simple-plot pilot — baseline (zero engagement)

Both variants, one loop — matches the pattern used everywhere else in this codebase (`generate_profile_posts.py`'s `VARIANTS` dict).

In [ ]:
for variant, post_text in VARIANTS.items():
    generated_count = skipped_count = 0
    for i in range(1, 101):
        POST_IMAGE_PATH = os.path.join(BASE_DIR, f"spotify_pie_plot/100_pie_charts_simple/spotify_genre_pie_chart_simple_{i:03d}.png")
        output_file = os.path.join(BASE_DIR, f"spotify_pie_plot/pie_plot_posts/baselines_simple_plot/{variant}/html/{i:03d}_remy_ashford_{SUFFIX[variant]}.html")

        # If the file already exists, skip generating it to save time
        if os.path.exists(output_file):
            skipped_count += 1
            continue

        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        html_content = generate_facebook_post(
            profile_name=PROFILE_NAME, post_text=post_text, post_time=POST_TIME,
            reactions=REACTIONS, comment_count=COMMENT_COUNT, share_count=SHARE_COUNT,
            post_image_path=POST_IMAGE_PATH, output_file=output_file,
            verified=VERIFIED, profile_image_path=PROFILE_IMAGE_PATH,
        )
        generated_count += 1
    print(f"  -> '{variant}': generated {generated_count}, skipped {skipped_count} (already existed)")


## Simple-plot pilot — metrics: realistic

In [ ]:
import importlib
import math
import random
import os
import post_generator_all_visible_emojis
importlib.reload(post_generator_all_visible_emojis)
from post_generator_all_visible_emojis import generate_facebook_post, display_facebook_post

PROFILE_NAME = "Remy Ashford"
POST_TIME = "Today at 2:43 PM"
VERIFIED = False
PROFILE_IMAGE_PATH = None
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000] # log scale
VARIANTS = {
    "incorrect": "The Spotify 2026 music genres distribution has just been released. Looks like Latin was more popular than Pop this year!",
    "correct": "The Spotify 2026 music genres distribution has just been released. Looks like Pop was more popular than Latin this year!",
}
SUFFIX = {"incorrect": "i", "correct": "c"}

REACTION_TYPES = ["like", "love", "haha", "wow", "sad", "angry"]

def make_log_reactions(scale_value, jitter=0.10, seed=None):
    rng = random.Random(seed)
    n = len(REACTION_TYPES)

    log_weights = [math.log(n + 1 - i) for i in range(n)]
    total_weight = sum(log_weights)
    shares = [w / total_weight for w in log_weights]

    # Shuffle the shares, not the reaction types
    rng.shuffle(shares)

    reactions = {}
    for emoji, share in zip(REACTION_TYPES, shares):
        base = scale_value * share
        factor = 1 + rng.uniform(-jitter, jitter)
        reactions[emoji] = max(1, round(base * factor))
    return reactions

for scale_value in REACTION_VALUES:
    generated_count = skipped_count = 0
    for variant, post_text in VARIANTS.items():
        for i in range(1, 101):
            output_file = os.path.join(BASE_DIR, f"spotify_pie_plot/pie_plot_posts/metrics_simple_plot/remy-ashford/{variant}/html/realistic/{scale_value}/{i:03d}_remy_ashford_{SUFFIX[variant]}.html")

            # If the file already exists, skip generating it to save time
            if os.path.exists(output_file):
                skipped_count += 1
                continue

            REACTIONS = make_log_reactions(scale_value, jitter=0.10, seed=i)
            COMMENT_COUNT = max(1, round(scale_value * 0.08))
            SHARE_COUNT   = max(1, round(scale_value * 0.04))

            POST_IMAGE_PATH = os.path.join(BASE_DIR, f"spotify_pie_plot/100_pie_charts_simple/spotify_genre_pie_chart_simple_{i:03d}.png")

            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            html_content = generate_facebook_post(
                profile_name=PROFILE_NAME,
                post_text=post_text,
                post_time=POST_TIME,
                reactions=REACTIONS,
                comment_count=COMMENT_COUNT,
                share_count=SHARE_COUNT,
                post_image_path=POST_IMAGE_PATH,
                output_file=output_file,
                verified=VERIFIED,
                profile_image_path=PROFILE_IMAGE_PATH,
            )
            generated_count += 1
    print(f"✅ Generated scale value: {scale_value} (generated {generated_count}, skipped {skipped_count})")
print("\n✅ All metric variants complete.")


## Simple-plot pilot — metrics: likes_only_noise

In [ ]:
import importlib
import os
import random
import csv
import post_generator_all_visible_emojis

importlib.reload(post_generator_all_visible_emojis)
from post_generator_all_visible_emojis import (
    generate_facebook_post,
    display_facebook_post,
)

# You must define your base directory here!

PROFILE_NAME = "Remy Ashford"
POST_TIME = "Today at 2:43 PM"
VERIFIED = False
PROFILE_IMAGE_PATH = None
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]
VARIANTS = {
    "incorrect": "The Spotify 2026 music genres distribution has just been released. Looks like Latin was more popular than Pop this year!",
    "correct": "The Spotify 2026 music genres distribution has just been released. Looks like Pop was more popular than Latin this year!",
}
SUFFIX = {"incorrect": "i", "correct": "c"}

JITTER_MAP = {
    10: 0.30,       
    100: 0.15,      
    1000: 0.15,     
    10000: 0.15,    
    100000: 0.15,   
    1000000: 0.15,  
}

# --- Initialize CSV Log File ---
csv_log_path = os.path.join(BASE_DIR, "spotify_pie_plot/pie_plot_posts/metrics_simple_plot/remy-ashford/correct/html/likes_only_noise/jitter_assignment_log_likes_only_noise.csv")
os.makedirs(os.path.dirname(csv_log_path), exist_ok=True)
# Only write the header if the file doesn't exist yet to avoid duplicate headers on restarts
if not os.path.exists(csv_log_path):
    with open(csv_log_path, mode='w', newline='') as log_file:
        writer = csv.writer(log_file)
        writer.writerow(["Image_Index", "Base_Scale", "Target_Margin", "Assigned_Likes", "Actual_Variance"])

for scale_value in REACTION_VALUES:
    COMMENT_COUNT = 0
    SHARE_COUNT = 0

    margin = JITTER_MAP[scale_value]
    low_bound = int(scale_value * (1 - margin))
    high_bound = int(scale_value * (1 + margin))

    possible_values = [
        val for val in range(max(1, low_bound), high_bound + 1) if val != scale_value
    ]
    
    if not possible_values:
        possible_values = [scale_value - 1, scale_value + 1] 

    jittered_values_for_scale = [0] + [random.choice(possible_values) for _ in range(100)]

    # --- Write to CSV Log ---
    with open(csv_log_path, mode='a', newline='') as log_file:
        writer = csv.writer(log_file)
        for idx in range(1, 101):
            assigned = jittered_values_for_scale[idx]
            actual_variance = f"{((assigned - scale_value) / scale_value) * 100:+.1f}%"
            writer.writerow([f"{idx:03d}", scale_value, f"±{margin*100}%", assigned, actual_variance])

    for variant, post_text in VARIANTS.items():
        skipped_count = 0
        generated_count = 0
        
        for i in range(1, 101):
            output_file = os.path.join(
                BASE_DIR,
                f"spotify_pie_plot/pie_plot_posts/metrics_simple_plot/remy-ashford/{variant}/html/likes_only_noise/{scale_value}/{i:03d}_remy_ashford_{SUFFIX[variant]}.html",
            )
            
            # --- OVERWRITE CHECK ---
            # If the file already exists, skip generating it to save time
            if os.path.exists(output_file):
                skipped_count += 1
                continue
            
            # If it doesn't exist, proceed with generation
            assigned_likes = jittered_values_for_scale[i]

            REACTIONS = {
                "like": assigned_likes,
                "love": 0,
                "haha": 0,
                "wow": 0,
                "sad": 0,
                "angry": 0,
            }

            POST_IMAGE_PATH = os.path.join(
                BASE_DIR,
                f"spotify_pie_plot/100_pie_charts_simple/spotify_genre_pie_chart_simple_{i:03d}.png",
            )

            # f"spotify_pie_plot/pie_visualizations/pop_23_5_latin_11/spotify_genre_pie_chart_{i:03d}.png",
            os.makedirs(os.path.dirname(output_file), exist_ok=True)

            html_content = generate_facebook_post(
                profile_name=PROFILE_NAME,
                post_text=post_text,
                post_time=POST_TIME,
                reactions=REACTIONS,
                comment_count=COMMENT_COUNT,
                share_count=SHARE_COUNT,
                post_image_path=POST_IMAGE_PATH,
                output_file=output_file,
                verified=VERIFIED,
                profile_image_path=PROFILE_IMAGE_PATH,
            )
            generated_count += 1

        print(f"  -> Variant '{variant}': Generated {generated_count}, Skipped {skipped_count} (already existed).")

    print(f"✅ Finished processing scale {scale_value}.")

print("\n✅ All metric variants successfully processed.")

## Phase 1 benchmarking — profile-variant posts (news outlets)

Already fully generated and tracked (`spotify_pie_plot/pie_plot_posts/for_benchmarking/`, 12/12 files) — kept as the historical record of how they were built, not something a reviewer needs to run.

In [ ]:
from post_generator_all_visible_emojis import generate_facebook_post, display_facebook_post

PROFILES = [
    {"name": "The New York Times", "slug": "ny_times", "verified": True, "profile_image_path": os.path.join(BASE_DIR, "news-profile-images/ny_times_profile.jpg")},
    {"name": "Fox News",           "slug": "fox_news", "verified": True, "profile_image_path": os.path.join(BASE_DIR, "news-profile-images/fox_news_profile.png")},
    {"name": "Reuters",            "slug": "reuters",  "verified": True, "profile_image_path": os.path.join(BASE_DIR, "news-profile-images/reuters_profile.jpg")},
]

POST_TEXTS = {
    "correct":   "The Spotify 2026 music genres distribution has just been released. Looks like Pop was more popular than Latin this year!",
    "incorrect": "The Spotify 2026 music genres distribution has just been released. Looks like Latin was more popular than Pop this year!",
}

POST_TIME = "Today at 2:43 PM"

REACTION_DEFS = {
    "like":  ("👍", "#1877f2"),
    "love":  ("❤️", "#f33e58"),
    "haha":  ("😆", "#f7b928"),
    "wow":   ("😮", "#f7b928"),
    "sad":   ("😢", "#f7b928"),
    "angry": ("😡", "#e9710f"),
}
REACTIONS = {k: 0 for k in REACTION_DEFS}
COMMENT_COUNT = 0
SHARE_COUNT   = 0

generated_count = skipped_count = 0
for label, post_text in POST_TEXTS.items():
    for profile in PROFILES:
        profile_slug = profile["slug"].lower().replace(" ", "_")
        suffix = "c" if label == "correct" else "i"

        for i in range(1, 101):
            output_file = os.path.join(BASE_DIR, f"spotify_pie_plot/pie_plot_posts/baselines_simple_plot/{label}/html/news/{i:03d}_{profile_slug}_{suffix}.html")

            # If the file already exists, skip generating it to save time
            if os.path.exists(output_file):
                skipped_count += 1
                continue

            POST_IMAGE_PATH = os.path.join(BASE_DIR, f"spotify_pie_plot/100_pie_charts_simple/spotify_genre_pie_chart_simple_{i:03d}.png")
            os.makedirs(os.path.dirname(output_file), exist_ok=True)  # creates folder in case it doesn't exist already

            html_content = generate_facebook_post(
                profile_name=profile["name"],
                post_text=post_text,
                post_time=POST_TIME,
                reactions=REACTIONS,
                comment_count=COMMENT_COUNT,
                share_count=SHARE_COUNT,
                post_image_path=POST_IMAGE_PATH,
                output_file=output_file,
                verified=profile["verified"],
                profile_image_path=profile["profile_image_path"],
            )
            generated_count += 1
print(f"generated {generated_count}, skipped {skipped_count} (already existed)")
